# # app
#
# > The web app: today, the session, the graph, search, and adding sources. Loopback only.

In [ ]:
#| default_exp app

In [ ]:
from fastcore.test import *

In [ ]:
#| export
import json, os, time, argparse
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from fasthtml.common import *
from fastcore.utils import *
from abhyasa.store import Learning
from abhyasa.model import DEPTHS, GOAL, recall, pick, learned
from abhyasa.tutor import Tutor, Session
from abhyasa.sources import extract, from_vault, from_vedicreader
from abhyasa.llm import rishi_ask

# ## Look
#
# Muted and readable: one column, large type, no boxes inside boxes. Light and dark follow the system.
# Keys use the left hand: ⌥D dictates, ⌥S sends, ⌥W asks for a hint.

In [ ]:
#| export
CSS = """
:root { --bg:#f7f6f2; --fg:#26241f; --mute:#77736a; --line:#e2ded4; --acc:#3d6b5a; --warn:#9a5b2e; --bad:#8f3b3b;
        --mono: ui-monospace, 'SF Mono', Menlo, monospace; color-scheme: light dark; }
@media (prefers-color-scheme: dark) { :root { --bg:#1b1a18; --fg:#e8e4da; --mute:#9a958a; --line:#34322d; --acc:#8fc2ab; --warn:#d6a073; --bad:#d68a8a; } }
* { box-sizing: border-box; }
body { background: var(--bg); color: var(--fg); font: 19px/1.6 ui-serif, 'Iowan Old Style', Georgia, serif; margin: 0; }
main { max-width: 760px; margin: 0 auto; padding: 28px 22px 80px; }
nav { display: flex; gap: 22px; align-items: baseline; padding: 14px 22px; border-bottom: 1px solid var(--line);
      font: 15px/1.4 -apple-system, system-ui, sans-serif; }
nav a { color: var(--mute); text-decoration: none; } nav a:hover, nav a.on { color: var(--fg); }
nav .brand { color: var(--fg); font-weight: 600; letter-spacing: .02em; }
nav form { margin-left: auto; } nav input { width: 220px; }
h1 { font-size: 30px; font-weight: 500; margin: 8px 0 18px; } h2 { font-size: 21px; font-weight: 600; margin: 34px 0 10px; }
a { color: var(--acc); }
.mute, .why, small { color: var(--mute); font: 14px/1.5 -apple-system, system-ui, sans-serif; }
.q { font-size: 24px; line-height: 1.5; margin: 6px 0 22px; }
label { display: block; font: 14px/1.4 -apple-system, system-ui, sans-serif; color: var(--mute); margin: 16px 0 6px; }
textarea, input[type=text], input[type=search], select { width: 100%; font: 18px/1.5 ui-serif, Georgia, serif; color: var(--fg);
      background: transparent; border: 0; border-bottom: 1px solid var(--line); padding: 8px 2px; outline: none; resize: vertical; }
textarea:focus, input:focus { border-bottom-color: var(--acc); }
input[type=range] { width: 100%; accent-color: var(--acc); }
.row { display: flex; gap: 12px; align-items: center; flex-wrap: wrap; margin-top: 18px; }
button, .btn { font: 15px/1 -apple-system, system-ui, sans-serif; padding: 11px 18px; border-radius: 8px; border: 1px solid var(--line);
      background: transparent; color: var(--fg); cursor: pointer; text-decoration: none; }
button.go { background: var(--acc); color: var(--bg); border-color: var(--acc); }
button:hover { border-color: var(--fg); }
kbd { font: 12px var(--mono); color: var(--mute); border: 1px solid var(--line); border-radius: 4px; padding: 1px 5px; margin-left: 6px; }
.opt { display: block; width: 100%; text-align: left; margin: 8px 0; font: 18px/1.5 ui-serif, Georgia, serif; padding: 12px 14px; }
.fb { margin-top: 26px; padding-top: 18px; border-top: 1px solid var(--line); }
.fb.ok .tag { color: var(--acc); } .fb.no .tag { color: var(--warn); } .tag { font: 600 13px -apple-system, system-ui, sans-serif; letter-spacing: .06em; text-transform: uppercase; }
.hints li { margin: 6px 0; color: var(--fg); } .hints { padding-left: 20px; }
.depth { display: inline-flex; gap: 3px; vertical-align: middle; margin-left: 6px; }
.depth i { width: 18px; height: 6px; border-radius: 3px; background: var(--line); display: inline-block; }
.depth i.f { background: var(--acc); } .depth i.a { background: var(--warn); }
ul.plain { list-style: none; padding: 0; } ul.plain li { padding: 9px 0; border-bottom: 1px solid var(--line); }
.mic.on { background: var(--bad); color: var(--bg); border-color: var(--bad); }
.htmx-indicator { opacity: 0; transition: opacity .2s; } .htmx-request .htmx-indicator, .htmx-request.htmx-indicator { opacity: 1; }
.spin { color: var(--mute); font: 14px -apple-system, system-ui, sans-serif; }
#graph { width: 100%; height: 72vh; } #graph text { font: 11px -apple-system, system-ui, sans-serif; fill: var(--mute); pointer-events: none; }
.wide { max-width: 1200px; }
pre, code { font-family: var(--mono); font-size: 15px; }
"""

# The browser side: markdown and KaTeX for questions, a WAV recorder for the mic, and the left-hand keys.

In [ ]:
#| export
JS = r"""
function render(root) {
  (root || document).querySelectorAll('.md:not([data-done])').forEach(el => {
    el.innerHTML = marked.parse(el.textContent); el.dataset.done = 1;
    if (window.renderMathInElement) renderMathInElement(el, {delimiters: [{left:'$$',right:'$$',display:true},{left:'$',right:'$',display:false}]});
  });
}
document.addEventListener('DOMContentLoaded', () => render());
document.addEventListener('htmx:afterSwap', e => render(e.target));

let rec = null;
async function mic(btn) {
  if (rec) { return rec.stop(); }
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const ctx = new AudioContext(), src = ctx.createMediaStreamSource(stream), node = ctx.createScriptProcessor(4096, 1, 1), bufs = [];
  node.onaudioprocess = e => bufs.push(new Float32Array(e.inputBuffer.getChannelData(0)));
  src.connect(node); node.connect(ctx.destination);
  btn.classList.add('on'); btn.firstChild.textContent = 'Stop ';
  rec = { stop: async () => {
    node.disconnect(); src.disconnect(); stream.getTracks().forEach(t => t.stop());
    btn.classList.remove('on'); btn.firstChild.textContent = 'Listening… '; rec = null;
    const wav = toWav(bufs, ctx.sampleRate); ctx.close();
    const r = await fetch('/stt?hint=' + encodeURIComponent(btn.dataset.hint || ''), {method: 'POST', body: wav});
    const t = (await r.json()).text || '';
    const box = document.getElementById(btn.dataset.into);
    box.value = (box.value ? box.value.trimEnd() + ' ' : '') + t; box.focus();
    btn.firstChild.textContent = 'Speak ';
  }};
}
function toWav(bufs, rate) {
  const n = bufs.reduce((a, b) => a + b.length, 0), ratio = rate / 16000, m = Math.floor(n / ratio), pcm = new Int16Array(m);
  const all = new Float32Array(n); let o = 0; bufs.forEach(b => { all.set(b, o); o += b.length; });
  for (let i = 0; i < m; i++) { const s = Math.max(-1, Math.min(1, all[Math.floor(i * ratio)])); pcm[i] = s < 0 ? s * 0x8000 : s * 0x7fff; }
  const h = new DataView(new ArrayBuffer(44)), w = (p, s) => [...s].forEach((c, i) => h.setUint8(p + i, c.charCodeAt(0)));
  w(0, 'RIFF'); h.setUint32(4, 36 + m * 2, true); w(8, 'WAVEfmt '); h.setUint32(16, 16, true); h.setUint16(20, 1, true); h.setUint16(22, 1, true);
  h.setUint32(24, 16000, true); h.setUint32(28, 32000, true); h.setUint16(32, 2, true); h.setUint16(34, 16, true); w(36, 'data'); h.setUint32(40, m * 2, true);
  return new Blob([h, pcm], {type: 'audio/wav'});
}
document.addEventListener('keydown', e => {
  if (!e.altKey) return;
  const k = {KeyD: '#mic', KeyS: '#send', KeyW: '#hint'}[e.code];
  const el = k && document.querySelector(k);
  if (el) { e.preventDefault(); el.click(); }
});
"""

HDRS = (Style(CSS), Script(src='https://cdn.jsdelivr.net/npm/marked@14/marked.min.js'),
        Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/katex@0.16/dist/katex.min.css'),
        Script(src='https://cdn.jsdelivr.net/npm/katex@0.16/dist/katex.min.js', defer=True),
        Script(src='https://cdn.jsdelivr.net/npm/katex@0.16/dist/contrib/auto-render.min.js', defer=True),
        Script(JS))

# ## Pieces

In [ ]:
#| export
def Md(s, cls='', **kw): return Div(s, cls=f'md {cls}'.strip(), **kw)

def DepthBar(floor, ceiling=6, asked=None, goal=GOAL):
    "Five marks: filled to the floor, the asked depth marked."
    return Span(*[I(cls='f' if d <= floor else 'a' if d == asked else '') for d in range(1, 6)], cls='depth',
                title=f'floor {DEPTHS[floor]}' + (f', asked at {DEPTHS[asked]}' if asked else ''))

def Page(title, *c, on='', wide=False):
    links = [('/', 'Today'), ('/graph', 'Graph'), ('/add', 'Add'), ('/concepts', 'Concepts')]
    return Title(f'{title} · abhyasa'), Nav(A('abhyasa', href='/', cls='brand'),
        *[A(t, href=h, cls='on' if t == on else '') for h, t in links],
        Form(Input(type='search', name='q', placeholder='Search what you learned'), action='/search')), Main(*c, cls='wide' if wide else '')

def Spin(txt='thinking…'): return Span(txt, cls='spin htmx-indicator')

def when(t):
    "A short, human time until `t`."
    if not t: return 'not scheduled'
    d = (t - time.time()) / 86400
    if d <= 0: return 'due now'
    return f'in {d*24:.0f} h' if d < 1 else f'in {d:.0f} d'

# ## The question card

In [ ]:
#| export
def QuestionCard(sid, q):
    "One question: why it was chosen, the question, and the answer form."
    head = P(f"{q.name} · {DEPTHS[q.depth]}", DepthBar(0, asked=q.depth), Br(), Span(q.why, cls='why'), cls='mute')
    if q.get('options'):
        body = Div(*[Button(Md(o), name='answer', value=str(i), cls='opt', hx_post=f'/s/{sid}/answer', hx_target='#fb',
                            hx_include='#conf', hx_indicator='#sp') for i, o in enumerate(q.options)],
                   Label('Confidence', Input(type='range', name='confidence', min=0, max=100, value=50, id='conf')),
                   Div(id='fb'), Spin(), id='sp')
        return Div(head, Md(q.question, cls='q'), body, id='card')
    form = Form(
        Label('Your answer', fr='ans'), Textarea(name='answer', id='ans', rows=3, autofocus=True),
        Label('Think aloud: how you got there, what you are unsure of', fr='think'),
        Textarea(name='thinking', id='think', rows=4),
        Div(Button('Speak ', Kbd('⌥D'), type='button', id='mic', cls='mic', onclick='mic(this)', data_into='think', data_hint=q.name),
            cls='row'),
        Label('Confidence', Input(type='range', name='confidence', min=0, max=100, value=50)),
        Div(Button('Send ', Kbd('⌥S'), cls='go', id='send'),
            Button('Hint ', Kbd('⌥W'), type='button', id='hint', hx_post=f'/s/{sid}/hint', hx_target='#hints', hx_swap='beforeend', hx_indicator='#sp'),
            Button("I don't know", type='button', hx_post=f'/s/{sid}/giveup', hx_vals='{"dont_know": 1}', hx_target='#fb', hx_indicator='#sp'),
            Spin(), cls='row', id='sp'),
        hx_post=f'/s/{sid}/answer', hx_target='#fb', hx_indicator='#sp')
    return Div(head, Md(q.question, cls='q'), Ol(id='hints', cls='hints'), form, Div(id='fb'), id='card')

def Feedback(sid, r):
    "The result of one answer: a nudge to try again, or the outcome, the lesson and what comes next."
    if not r.done:
        return Div(Span('Not yet', cls='tag'), Md(r.feedback), P(f'{r.tries_left} more tries. Or ',
                   A('teach me', href='#', hx_post=f'/s/{sid}/giveup', hx_target='#fb'), '.', cls='mute'), cls='fb no')
    ok = r.correct
    return Div(Span({'solid': 'Solid', 'fragile': 'Right, but shaky', 'misconception': 'A misconception', 'gap': 'A gap'}[r.outcome], cls='tag'),
               Md(r.feedback) if r.feedback else '', Md(r.lesson) if r.lesson else '',
               P(f"Now: floor {DEPTHS[r.floor]}, next review {when(r.due)}.", cls='mute'),
               Div(Button('Next ', Kbd('⌥S'), cls='go', id='send', hx_post=f'/s/{sid}/next', hx_target='#card', hx_swap='outerHTML',
                          hx_indicator='#sp2', autofocus=True),
                   A('End session', href=f'/s/{sid}/end', cls='btn'), Spin(), cls='row', id='sp2'),
               cls=f"fb {'ok' if ok else 'no'}")

# ## The app

In [ ]:
#| export
def mk_app(learning:Learning, ask=None, transcribe=None):
    "The FastHTML app over `learning`; `ask` is the model, `transcribe` the speech to text."
    from abhyasa.voice import transcribe as _tx
    ask, transcribe = ask or rishi_ask(), transcribe or _tx
    tutor, sessions, pre = Tutor(learning, ask), {}, {}
    pool = ThreadPoolExecutor(2)
    app, rt = fast_app(hdrs=HDRS, pico=False, default_hdrs=True)
    app.state.learning, app.state.tutor, app.state.sessions = learning, tutor, sessions

    def nxt(sid):
        "The next question: the one fetched ahead if there is one."
        s = sessions[sid]
        f = pre.pop(sid, None)
        return f.result() if f else s.next()

    @rt('/')
    def get(subject:str=''):
        cs = learning.concepts()
        now = time.time()
        due = [c for c in cs if c.seen and c.due and c.due <= now]
        subs = learning.subjects()
        p = pick(cs, subject=subject or None)
        start = Form(Input(type='hidden', name='subject', value=subject),
                     Button(f"Start{' ' + subject if subject else ''}", cls='go'), action='/start', method='post')
        return Page('Today',
            H1('Today'),
            P(f"{len(due)} due for review · {sum(learned(c) for c in cs)} of {len(cs)} concepts learned", cls='mute'),
            P(f"Next: {p[0].name} at {DEPTHS[p[1]]}. ", Span(p[2], cls='why')) if p else P('Nothing due. Add a source, or pick a concept.'),
            start,
            H2('Subjects'),
            Ul(*[Li(A(r['subject'] or 'general', href=f"/?subject={r['subject'] or ''}"), Span(f"  {r['n']} concepts", cls='mute')) for r in subs], cls='plain'),
            on='Today')

    @rt('/start')
    def post(subject:str='', focus:str=''):
        s = Session(tutor, focus=focus or None, subject=subject or None)
        sessions[s.id] = s
        return Redirect(f'/s/{s.id}')

    @rt('/s/{sid}')
    def get(sid:str):
        s = sessions.get(sid)
        if not s: return Redirect('/')
        q = s.q if s.q and s.q.id in tutor.open else nxt(sid)
        if not q: return Redirect(f'/s/{sid}/end')
        return Page('Learn', QuestionCard(sid, q))

    @rt('/s/{sid}/answer')
    def post(sid:str, answer:str='', thinking:str='', confidence:int=50):
        s = sessions[sid]
        r = s.answer(answer, thinking, confidence)
        if r.done: pre[sid] = pool.submit(s.next)
        return Feedback(sid, r)

    @rt('/s/{sid}/giveup')
    def post(sid:str, dont_know:int=0, confidence:int=0):
        s = sessions[sid]
        r = s.give_up(dont_know=bool(dont_know), confidence=confidence) if not dont_know else s.answer(dont_know=True)
        pre[sid] = pool.submit(s.next)
        return Feedback(sid, r)

    @rt('/s/{sid}/hint')
    def post(sid:str): return Li(Md(sessions[sid].hint()))

    @rt('/s/{sid}/next')
    def post(sid:str):
        q = nxt(sid)
        if not q: return Div(H1('Done for now'), P(sessions[sid].summary()), A('Back to today', href='/'), id='card')
        return QuestionCard(sid, q)

    @rt('/s/{sid}/end')
    def get(sid:str):
        s = sessions.pop(sid, None)
        if (f := pre.pop(sid, None)): f.cancel()
        return Page('Session', H1('Session'), P(s.summary() if s else 'This session is over.'), A('Back to today', href='/'))

    @rt('/stt')
    async def post(req, hint:str=''):
        audio = await req.body()
        try: return JSONResponse(dict(text=transcribe(audio, hint=hint)))
        except Exception as e: return JSONResponse(dict(text='', error=str(e)), status_code=500)

    @rt('/concepts')
    def get(subject:str=''):
        cs = sorted(learning.concepts(subject or None), key=lambda c: (c.subject, c.get('order', 1e9), c.name))
        return Page('Concepts', H1('Concepts'), Ul(*[Li(A(c.name, href=f'/c/{c.slug}'), DepthBar(c.floor),
                    Span(f"  {c.subject} · {when(c.due) if c.seen else 'new'}", cls='mute')) for c in cs], cls='plain'), on='Concepts')

    @rt('/c/{slug}')
    def get(slug:str):
        c = learning.concept(slug)
        if not c: return Page('Concept', P('No such concept.'))
        by = {x.slug: x for x in learning.concepts()}
        refs = lambda ss: Ul(*[Li(A(by[s].name, href=f'/c/{s}'), DepthBar(by[s].floor)) for s in ss if s in by], cls='plain') if ss else P('none', cls='mute')
        att = learning.attempts(slug, limit=10)
        return Page(c.name, H1(c.name), P(c.subject, ' · floor ', DEPTHS[c.floor], DepthBar(c.floor), ' · ',
                    f"recall {recall(c):.0%}" if c.seen else 'new', ' · ', when(c.due) if c.seen else '', cls='mute'),
            Md(c.statement or ''),
            Form(Input(type='hidden', name='focus', value=slug), Button('Learn this', cls='go'), action='/start', method='post'),
            *([H2('Misconceptions seen'), Ul(*map(Li, c.misconceptions))] if c.misconceptions else []),
            H2('Needs first'), refs(c.prereqs), H2('Related'), refs(c.related),
            H2('Your attempts'), Ul(*[Li(Md(a.question), P(('✓ ' if a.correct else '✗ ') + (a.answer or '—'), cls='mute')) for a in att], cls='plain')
            if att else P('none yet', cls='mute'),
            *([H2('Source'), Ul(*[Li(A(p.get('title') or p.get('lesson') or p.get('src'), href=p['url']) if p.get('url') else
                                     (p.get('title') or p.get('term') or p.get('key') or p.get('src'))) for p in c.provenance], cls='plain')] if c.provenance else []))

    @rt('/graph')
    def get(subject:str=''):
        return Page('Graph', Div(Select(Option('all subjects', value=''), *[Option(r['subject'], selected=r['subject'] == subject)
                    for r in learning.subjects() if r['subject']], onchange="location='/graph?subject='+this.value"), cls='row'),
                    Svg(id='graph'), Script(src='https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js'),
                    Script(GRAPH_JS.replace('__URL__', f'/graph.json?subject={subject}')), on='Graph', wide=True)

    @rt('/graph.json')
    def get(subject:str=''): return JSONResponse(learning.graph(subject or None))

    @rt('/search')
    def get(q:str=''):
        hs = learning.search(q, limit=12) if q else []
        try: el = learning.v.elsewhere(q, limit=2, max_read=300) if q else []
        except Exception: el = []
        return Page('Search', H1(f'“{q}”'), H2('In what you learned'),
            Ul(*[Li(Md(h['snippet']), Span(h['breadcrumb'], cls='mute')) for h in hs], cls='plain') if hs else P('nothing', cls='mute'),
            H2('In your vault'), Ul(*[Li(Md(h.text[:300]), Span(h.breadcrumb, cls='mute')) for h in el], cls='plain') if el else P('nothing', cls='mute'))

    @rt('/add')
    def get():
        return Page('Add', H1('Add a source'),
            P('Paste text, or give a vault document (id, source or title). The tutor finds its core ideas.', cls='mute'),
            Form(Label('Subject', Input(type='text', name='subject', placeholder='e.g. ml, sanskrit, philosophy')),
                 Label('Text', Textarea(name='text', rows=8)), Label('or a vault document', Input(type='text', name='ref')),
                 Div(Button('Find concepts', cls='go'), Spin('reading…'), cls='row'),
                 hx_post='/add', hx_target='#out', hx_indicator='.spin'),
            Div(id='out'),
            H2('vedicreader'), P('Sanskrit lessons and terms, and the jyotiṣa grahas, rāśis and nakṣatras, from the repo files.', cls='mute'),
            Button('Import vedicreader', hx_post='/add/vedicreader', hx_target='#vr'), Div(id='vr'), on='Add')

    @rt('/add')
    def post(subject:str='', text:str='', ref:str=''):
        slugs = from_vault(learning, ask, ref, subject) if ref else extract(learning, ask, text, subject, dict(src='pasted', at=time.time()))
        return Ul(*[Li(A(learning.concept(s).name, href=f'/c/{s}')) for s in slugs], cls='plain')

    @rt('/add/vedicreader')
    def post(): return P(', '.join(f'{v} {k}' for k, v in from_vedicreader(learning).items()) + ' added.')

    return app

# ## The graph
#
# A force layout: colour by floor, a ring when a review is due, solid links for prereqs (arrow into the later concept), dashed
# for related. Click a node to open it.

In [ ]:
#| export
GRAPH_JS = r"""
(async () => {
  const g = await (await fetch('__URL__')).json(), svg = d3.select('#graph'), el = svg.node();
  const W = el.clientWidth, H = el.clientHeight, now = Date.now() / 1000;
  const css = getComputedStyle(document.documentElement), v = n => css.getPropertyValue(n).trim();
  const col = d3.scaleLinear().domain([0, 5]).range([v('--line'), v('--acc')]);
  svg.append('defs').append('marker').attr('id', 'ar').attr('viewBox', '0 -4 8 8').attr('refX', 16).attr('markerWidth', 6)
     .attr('markerHeight', 6).attr('orient', 'auto').append('path').attr('d', 'M0,-4L8,0L0,4').attr('fill', v('--mute'));
  const root = svg.append('g');
  svg.call(d3.zoom().scaleExtent([.2, 4]).on('zoom', e => root.attr('transform', e.transform)));
  const sim = d3.forceSimulation(g.nodes).force('link', d3.forceLink(g.links).id(d => d.id).distance(60))
    .force('charge', d3.forceManyBody().strength(-120)).force('center', d3.forceCenter(W / 2, H / 2)).force('x', d3.forceX(W/2).strength(.03)).force('y', d3.forceY(H/2).strength(.03));
  const link = root.append('g').selectAll('line').data(g.links).join('line').attr('stroke', v('--line'))
    .attr('stroke-dasharray', d => d.kind === 'related' ? '3,3' : null).attr('marker-end', d => d.kind === 'prereq' ? 'url(#ar)' : null);
  const node = root.append('g').selectAll('circle').data(g.nodes).join('circle').attr('r', d => 5 + Math.min(d.seen, 8))
    .attr('fill', d => col(d.floor)).attr('stroke', d => d.seen && d.due && d.due <= now ? v('--warn') : v('--mute')).attr('stroke-width', d => d.seen && d.due <= now ? 2.5 : .5)
    .style('cursor', 'pointer').on('click', (e, d) => location = '/c/' + d.id)
    .call(d3.drag().on('start', (e, d) => { if (!e.active) sim.alphaTarget(.3).restart(); d.fx = d.x; d.fy = d.y; })
      .on('drag', (e, d) => { d.fx = e.x; d.fy = e.y; }).on('end', (e, d) => { if (!e.active) sim.alphaTarget(0); d.fx = d.fy = null; }));
  node.append('title').text(d => `${d.name} (${d.subject}) floor ${d.floor}`);
  const label = root.append('g').selectAll('text').data(g.nodes).join('text').text(d => d.name).attr('dx', 9).attr('dy', 4);
  sim.on('tick', () => {
    link.attr('x1', d => d.source.x).attr('y1', d => d.source.y).attr('x2', d => d.target.x).attr('y2', d => d.target.y);
    node.attr('cx', d => d.x).attr('cy', d => d.y); label.attr('x', d => d.x).attr('y', d => d.y);
  });
})();
"""

# ## Run it
#
# `abhyasa` serves on 127.0.0.1:5077. `--vault` (or `$ABHYASA_VAULT`) names the vault file; the default is vishalakshi's.

In [ ]:
#| export
def main():
    "Serve the app on loopback."
    import uvicorn
    p = argparse.ArgumentParser(description='abhyasa: a learning harness that makes you think')
    p.add_argument('--port', type=int, default=int(os.getenv('ABHYASA_PORT', 5077)))
    p.add_argument('--vault', default=os.getenv('ABHYASA_VAULT') or os.getenv('VISHALAKSHI_VAULT'))
    p.add_argument('--model', default=None, help='a rishi model name; default $ABHYASA_MODEL or claude/sonnet')
    a = p.parse_args()
    lr = Learning(a.vault)
    print(f'abhyasa on http://127.0.0.1:{a.port}  ({lr})')
    uvicorn.run(mk_app(lr, rishi_ask(a.model)), host='127.0.0.1', port=a.port, log_level='warning')

# ## Tests

In [ ]:
from starlette.testclient import TestClient
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml')
fake = Fake({'You grade one answer': lambda p, sp: dict(correct='rank' in p.split("Learner's answer:")[1], grade=8, first_error='Not quite.',
                                                         misconception='', feedback='Look at positions.'),
             'Give hint level': 'Think about positions.', 'Teach the smallest step': 'Positions. Check: what if k = 0?'},
            dflt=dict(question='What does a rank tell you?', answer='Its position.', rubric=['position']))
cl = TestClient(mk_app(lr, fake, transcribe=lambda a, hint='': f'{len(a)} bytes'))
assert 'Ranking' in cl.get('/').text
r = cl.post('/start', data=dict(subject=''), follow_redirects=False)
sid = r.headers['location'].split('/')[-1]
page = cl.get(f'/s/{sid}').text
assert 'What does a rank tell you?' in page and 'Its position' not in page    # the model answer stays hidden
assert 'Not yet' in cl.post(f'/s/{sid}/answer', data=dict(answer='a score', confidence=70)).text
assert 'Think about positions' in cl.post(f'/s/{sid}/hint').text
assert 'shaky' in cl.post(f'/s/{sid}/answer', data=dict(answer='its rank position', confidence=70)).text
assert 'What does a rank tell you?' in cl.post(f'/s/{sid}/next').text
test_eq(cl.post('/stt', content=b'1234').json(), dict(text='4 bytes'))
test_eq(len(cl.get('/graph.json').json()['nodes']), 1)
assert cl.get('/c/ranking').status_code == 200 and 'Your attempts' in cl.get('/c/ranking').text
assert cl.get('/search?q=rank').status_code == 200

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()